In [3]:
import pandas as pd
from mp_api.client import MPRester

df = pd.read_parquet("../data/processed/li_electrodes_clean.parquet")
print(f"{len(df)} clean electrodes to match")

# Fetch only the structure, keyed by battery_id
with MPRester() as mpr:
    docs = mpr.materials.insertion_electrodes.search(
        working_ion="Li",
        fields=["battery_id", "host_structure"],
    )
print(f"Fetched {len(docs)} structures from the API")

2634 clean electrodes to match


Retrieving InsertionElectrodeDoc documents:   0%|          | 0/2774 [00:00<?, ?it/s]

Fetched 2774 structures from the API


In [4]:
structures = {doc.battery_id: doc.host_structure for doc in docs}

clean_ids = set(df["battery_id"])
have = clean_ids & set(structures.keys())
missing = clean_ids - set(structures.keys())

print(f"Clean materials:         {len(clean_ids)}")
print(f"Have a structure:        {len(have)}")
print(f"Missing a structure:     {len(missing)}")

# Confirm what a structure object actually is
example = structures[df['battery_id'].iloc[0]]
print(f"\nType of host_structure: {type(example)}")
print(example)

Clean materials:         2634
Have a structure:        2634
Missing a structure:     0

Type of host_structure: <class 'pymatgen.core.structure.Structure'>
Full Formula (Ag1)
Reduced Formula: Ag
abc   :   2.941952   2.941952   2.941952
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (1)
  #  SP      a    b    c
---  ----  ---  ---  ---
  0  Ag      0    0    0


In [5]:
from monty.serialization import dumpfn

# Keep only the structures matching our clean dataset, in order
structures_clean = {bid: structures[bid] for bid in df["battery_id"]}
dumpfn(structures_clean, "../data/processed/host_structures.json.gz")
print(f"Saved {len(structures_clean)} structures")

Saved 2634 structures


In [6]:
from monty.serialization import loadfn

structures = loadfn("../data/processed/host_structures.json.gz")
print(f"Reloaded {len(structures)} structures, type: {type(next(iter(structures.values())))}")

# Find a representative multi-element example (an oxide with several atoms)
for bid, s in structures.items():
    elems = [str(e) for e in s.composition.elements]
    if len(s) >= 6 and "O" in elems and len(elems) >= 2:
        example_id, example = bid, s
        break

print(f"\n{example_id}: {example.composition.reduced_formula}, {len(example)} sites")
print(example)

Reloaded 2634 structures, type: <class 'pymatgen.core.structure.Structure'>

mp-1045514_Li: CrO2, 12 sites
Full Formula (Cr4 O8)
Reduced Formula: CrO2
abc   :   5.758493   5.758493   6.145182
angles:  57.752634  57.752643  60.052166
pbc   :       True       True       True
Sites (12)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Cr    0.006427  0.006427  0.993573
  1  Cr    0.243573  0.243573  0.256427
  2  Cr    0.625     0.625     0.125
  3  Cr    0.625     0.625     0.625
  4  O     0.371551  0.371551  0.352827
  5  O     0.392903  0.856765  0.375166
  6  O     0.371551  0.371551  0.904072
  7  O     0.856765  0.392903  0.375166
  8  O     0.393235  0.857097  0.874834
  9  O     0.857097  0.393235  0.874834
 10  O     0.878449  0.878449  0.345928
 11  O     0.878449  0.878449  0.897173


In [7]:
r = 6.0  # cutoff radius in Ångström
centers, neighbors, offsets, distances = example.get_neighbor_list(r)

print(f"Atoms (nodes):       {len(example)}")
print(f"Edges (pairs < {r} Å): {len(centers)}")
print(f"Avg neighbors/atom:  {len(centers) / len(example):.1f}")
print(f"Distance range:      {distances.min():.2f} – {distances.max():.2f} Å")

Atoms (nodes):       12
Edges (pairs < 6.0 Å): 932
Avg neighbors/atom:  77.7
Distance range:      1.71 – 5.96 Å


In [8]:
for r_test in [2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]:
    c, n, o, d = example.get_neighbor_list(r_test)
    print(f"r = {r_test:.1f} Å  ->  {len(c)/len(example):5.1f} neighbors/atom   ({len(c)} edges)")

r = 2.0 Å  ->    2.0 neighbors/atom   (24 edges)
r = 2.5 Å  ->    3.3 neighbors/atom   (40 edges)
r = 3.0 Å  ->   10.0 neighbors/atom   (120 edges)
r = 3.5 Å  ->   17.7 neighbors/atom   (212 edges)
r = 4.0 Å  ->   23.0 neighbors/atom   (276 edges)
r = 5.0 Å  ->   45.0 neighbors/atom   (540 edges)
r = 6.0 Å  ->   77.7 neighbors/atom   (932 edges)


In [9]:
import numpy as np

def capped_neighbors(structure, r=6.0, max_neighbors=12):
    """Return edges (src, dst, distance) keeping each atom's nearest
    `max_neighbors` within radius `r`, periodic images included."""
    centers, neighbors, offsets, distances = structure.get_neighbor_list(r)
    src, dst, edist = [], [], []
    for i in range(len(structure)):
        mask = centers == i
        nbr = neighbors[mask]
        d = distances[mask]
        order = np.argsort(d)[:max_neighbors]   # nearest N
        src.extend([i] * len(order))
        dst.extend(nbr[order])
        edist.extend(d[order])
    return np.array(src), np.array(dst), np.array(edist)

src, dst, dist = capped_neighbors(example, r=6.0, max_neighbors=12)
print(f"edges: {len(src)}   neighbors/atom: {len(src)/len(example):.1f}")
print(f"distance range: {dist.min():.2f} – {dist.max():.2f} Å")

edges: 144   neighbors/atom: 12.0
distance range: 1.71 – 3.40 Å
